# NB1 — Delta Lake Basics

**Mục tiêu:** Tạo Delta table, observe transaction log, demo schema enforcement.

Maps to slide §2 (Delta Lake) + deliverable bullet 1.

In [1]:
import sys
sys.path.append("/workspace/scripts")
from spark_session import get_spark, reset_path
from pyspark.sql import functions as F

spark = get_spark("nb1_delta_basics")

## 1. Write a Delta table

In [2]:
data = [
    (1, "alice", 30, "Hanoi"),
    (2, "bob", 25, "HCMC"),
    (3, "charlie", 35, "Danang"),
]
df = spark.createDataFrame(data, ["id", "name", "age", "city"])
table_path = "s3a://lakehouse/users_delta"
reset_path(spark, table_path)
df.write.format("delta").mode("overwrite").save(table_path)

## 2. Read it back + inspect transaction log

Open MinIO console (http://localhost:9001) → `lakehouse/users_delta/_delta_log/`.
You should see `00000000000000000000.json`.

In [3]:
spark.read.format("delta").load(table_path).show()
spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`").show(truncate=False)

+---+-------+---+------+
| id|   name|age|  city|
+---+-------+---+------+
|  3|charlie| 35|Danang|
|  1|  alice| 30| Hanoi|
|  2|    bob| 25|  HCMC|
+---+-------+---+------+



+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|0      |2026-06-22 05:15:24|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |false        |{numFiles -> 4, numOutputRows -> 3

## 3. Schema enforcement — try to write a wrong schema

In [4]:
try:
    bad = spark.createDataFrame([(4, "dan", "thirty", "Hue")], ["id", "name", "age", "city"])
    bad.write.format("delta").mode("append").save(table_path)
except Exception as e:
    print("BLOCKED by schema enforcement (expected):")
    print(type(e).__name__, str(e)[:200])

BLOCKED by schema enforcement (expected):
AnalysisException [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'age' and 'age'


## 4. Schema evolution (opt-in)

In [5]:
new_col = spark.createDataFrame(
    [(4, "dan", 28, "Hue", "premium")],
    ["id", "name", "age", "city", "tier"],
)
new_col.write.format("delta").mode("append").option("mergeSchema", "true").save(table_path)
spark.read.format("delta").load(table_path).show()

+---+-------+---+------+-------+
| id|   name|age|  city|   tier|
+---+-------+---+------+-------+
|  4|    dan| 28|   Hue|premium|
|  3|charlie| 35|Danang|   NULL|
|  1|  alice| 30| Hanoi|   NULL|
|  2|    bob| 25|  HCMC|   NULL|
+---+-------+---+------+-------+



## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] mergeSchema added the `tier` column

In [6]:
spark.stop()